In [ ]:
# Jupyter Notebook

# Importando as bibliotecas necessárias
import os
import json
import subprocess
import statsmodels
import pandas as pd
import seaborn as sns
from pathlib import Path
from matplotlib.patches import Patch
import numpy as np
import matplotlib.pyplot as plt
import EcoSimpy

# Configurações iniciais
sns.set_theme(style="whitegrid")
pd.options.mode.chained_assignment = None  # Para evitar warnings de cópias de DataFrame

# Configurando os diretórios
input_dir = Path("runs")
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)


## Execução da Simulação

In [ ]:
EcoSimpy.make_config_json()

config_json_file = "config.json"
scenario_file = "zero_det_scn_base_05.json"
model_file = "model_zero_det_base_05.json"

app_dir = str(Path.cwd())
print(f"Current directory: {app_dir}")

new_sim = EcoSimpy.Simulation(app_dir,
                              config_json_file,
                              model_file,
                              scenario_file,
							  clean_run = True)

new_sim.execute_simulation()


## Análise do Resultado

In [ ]:
# Configurações iniciais
sns.set(style="whitegrid")
pd.options.mode.chained_assignment = None  # Evita warnings desnecessários

# Config the paths
input_dir = Path("runs")

output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

# CSV
csv_files = list(input_dir.glob("*.csv"))

# Verificando se existem arquivos CSV
if not csv_files:
    raise ValueError(f"Nenhum arquivo CSV foi encontrado em: {input_dir.resolve()}")

# Gerando o DataFrame consolidado
runs_df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)
runs_df_or = runs_df.copy()
runs_df = runs_df.dropna()

# Calculando a diferença de payoffs
runs_df["diff_payoffs"] = runs_df["my_payoff"] - runs_df["other_payoff"]

# Agrupando os dados para calcular médias e somas de payoffs
sum_runs = (
    runs_df.groupby(["scenario", "step", "strategy_name"])
    .agg(
        payoff_mean=("my_payoff", "mean"),
        payoff_sum=("my_payoff", "sum"),
    )
    .reset_index()
)

# Calculando cooperação e defecção
sum_cooperation = (
    runs_df.groupby(["scenario", "step"])
    .agg(
        my_play_C=("my_play", lambda x: (x == "C").sum()),
        my_play_D=("my_play", lambda x: (x == "D").sum()),
        ot_play_C=("other_play", lambda x: (x == "C").sum()),
        ot_play_D=("other_play", lambda x: (x == "D").sum()),
        players=("my_play", "count"),
    )
    .reset_index()
)

sum_cooperation["cooperation"] = sum_cooperation["my_play_C"] / sum_cooperation["players"]
sum_cooperation["defection"] = sum_cooperation["my_play_D"] / sum_cooperation["players"]

# Tabela com média e mediana por agente
agent_summary = (
    runs_df.groupby("strategy_name")
    .agg(
        payoff_mean=("my_payoff", "mean"),
        payoff_median=("my_payoff", "median"),
        payoff_sum=("my_payoff", "sum"),
        observations=("my_payoff", "count"),
    )
    .reset_index()
    .sort_values(by="payoff_mean", ascending=False)
)

# Exibindo a tabela
print("Tabela de média e mediana por agente:")
display(agent_summary)

# Salvando a tabela
agent_summary.to_csv(output_dir / "agent_summary.csv", index=False)

# Gera uma paleta mais saturada
unique_strategies = sum_runs["strategy_name"].unique()
palette = sns.color_palette("bright", n_colors=len(unique_strategies))

# Cria um mapeamento de cores
color_dict = dict(zip(unique_strategies, palette))

# Gráfico 1: Payoff médio por estratégia ao longo do tempo
g = sns.lmplot(
    data=sum_runs, 
    x="step", 
    y="payoff_mean",
    hue="strategy_name",
    palette=color_dict,
    scatter=True, 
    lowess=True, 
    line_kws={"alpha": 0.7},
    scatter_kws={"alpha": 0.1, "s": 10},
    height=6, aspect=1.5
)

# Títulos e eixos
g.set_axis_labels("Passo", "Payoff Médio")
g.fig.suptitle("Payoff Médio por Estratégia ao Longo do Tempo", fontsize=14, y=1.03)

# Remove a legenda original
if g._legend is not None:
    g._legend.remove()

# Cria legenda personalizada com patches coloridos
handles = [Patch(color=color_dict[strategy], label=strategy) for strategy in unique_strategies]
g.ax.legend(handles=handles, title="Estratégia", fontsize=10, title_fontsize=11)

# Salvar e mostrar
plt.savefig(output_dir / "payoff_mean_by_strategy.png", bbox_inches="tight")
plt.show()

# Gráfico 2: Distribuição de payoff médio por estratégia
plt.figure(figsize=(10, 6))
sns.violinplot(data=sum_runs, x="strategy_name", y="payoff_mean", inner=None)
sns.boxplot(data=sum_runs, x="strategy_name", y="payoff_mean", width=0.2, color="white")
sns.pointplot(
    data=sum_runs,
    x="strategy_name",
    y="payoff_mean",
    color="red",
    join=False,
    markers="o",
    scale=1.5,
)
plt.title("Distribuição de Payoff Médio por Estratégia")
plt.xlabel("Estratégia")
plt.ylabel("Payoff Médio")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(output_dir / "payoff_distribution_by_strategy.png")
plt.show()

# Gráfico 3A: Cooperação ao longo do tempo (dispersão)
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=sum_cooperation, 
    x="step", 
    y="cooperation", 
    alpha=0.5
)
plt.title("Cooperação ao Longo do Tempo")
plt.xlabel("Passo")
plt.ylabel("Cooperação")
plt.savefig(output_dir / "cooperation_scatter.png")
plt.show()

# Gráfico 3B: Cooperação ao longo do tempo com suavização
g = sns.lmplot(
    data=sum_cooperation, 
    x="step", 
    y="cooperation", 
    scatter=True, 
    lowess=True, 
    line_kws={"alpha": 0.5},
    scatter_kws={"alpha": 0.1, "s": 10},
    height=6, aspect=1.5
)

g.set_axis_labels("Passo", "Cooperação")
g.fig.suptitle("Cooperação ao Longo do Tempo", fontsize=14, y=1.03)

plt.savefig(output_dir / "cooperation_over_time.png", bbox_inches="tight")
plt.show()


# Gráfico 4: Payoff médio normalizado versus cooperação
# Agrupa o payoff médio por passo e normaliza para o intervalo [-1, 1]





In [ ]:

# =========================
# 1. Preparação dos dados
# =========================
# CSV
csv_files = list(input_dir.glob("*.csv"))

# Gerando o DataFrame consolidado
runs_df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)
runs_df_or = runs_df.copy()
runs_df = runs_df.dropna().copy()

# Diferença de payoffs
runs_df["diff_payoffs"] = runs_df["my_payoff"] - runs_df["other_payoff"]

# =========================
# 2. Resumo por estratégia
# =========================

strategy_summary = (
    runs_df.groupby("strategy_name")
    .agg(
        payoff_mean=("my_payoff", "mean"),
        payoff_median=("my_payoff", "median"),
        payoff_sum=("my_payoff", "sum"),
        observations=("my_payoff", "count"),
        coop_count=("my_play", lambda x: (x == "C").sum()),
        defect_count=("my_play", lambda x: (x == "D").sum()),
    )
    .reset_index()
)

# Taxa de cooperação em [0, 1]
strategy_summary["coop_rate"] = (
    strategy_summary["coop_count"] / strategy_summary["observations"]
)

# Eixo X: normalização da cooperação para [-1, 1]
# 1   = sempre coopera
# -1  = nunca coopera
strategy_summary["coop_norm_m1_1"] = 2 * strategy_summary["coop_rate"] - 1

# =========================
# 3. Eixo Y normalizado
# =========================

# -------- Opção A --------
# Normalização com base no menor e maior payoff MÉDIO observado nos dados
y_min_obs = strategy_summary["payoff_mean"].min()
y_max_obs = strategy_summary["payoff_mean"].max()

if y_max_obs != y_min_obs:
    strategy_summary["payoff_norm_0_1"] = (
        (strategy_summary["payoff_mean"] - y_min_obs) / (y_max_obs - y_min_obs)
    )
else:
    strategy_summary["payoff_norm_0_1"] = 0.5


# -------- Opção B --------
# Se você quiser trocar o eixo Y para uma escala teórica do IPD, altere aqui:
#
# Exemplo 1: se quiser considerar que os payoffs médios relevantes vão de 1 a 3:
# ipd_min = 1
# ipd_max = 3
#
# Exemplo 2: se quiser considerar a escala total clássica do IPD de 0 a 5:
# ipd_min = 0
# ipd_max = 5
#
# Depois substitua a coluna do gráfico por esta:
#
# strategy_summary["payoff_norm_0_1"] = (
#     (strategy_summary["payoff_mean"] - ipd_min) / (ipd_max - ipd_min)
# ).clip(0, 1)
#
# O .clip(0, 1) garante que qualquer valor fora do intervalo fique truncado
# entre 0 e 1.


# =========================
# 4. Ordenando e exibindo
# =========================

strategy_summary = strategy_summary.sort_values(
    by="payoff_mean", ascending=False
).reset_index(drop=True)

print("Tabela resumo por estratégia:")
display(strategy_summary)

# Salvando a tabela
strategy_summary.to_csv(output_dir / "strategy_summary_scatter.csv", index=False)

# =========================
# 5. Paleta de cores
# =========================

unique_strategies = strategy_summary["strategy_name"].unique()
palette = sns.color_palette("bright", n_colors=len(unique_strategies))
color_dict = dict(zip(unique_strategies, palette))

# =========================
# 6. Gráfico
# =========================

plt.figure(figsize=(10, 7))

for _, row in strategy_summary.iterrows():
    plt.scatter(
        row["coop_norm_m1_1"],
        row["payoff_norm_0_1"],
        s=120,
        color=color_dict[row["strategy_name"]],
        alpha=0.85,
        edgecolor="black"
    )
    
    plt.text(
        row["coop_norm_m1_1"] + 0.02,
        row["payoff_norm_0_1"] + 0.02,
        row["strategy_name"],
        fontsize=9
    )

plt.axvline(0, linestyle="--", linewidth=1)
plt.ylim(-0.05, 1.05)
plt.xlim(-1.05, 1.05)

plt.xlabel("Cooperação da estratégia (-1 = nunca coopera, 1 = sempre coopera)")
plt.ylabel("Payoff médio normalizado (0 a 1)")
plt.title("Cooperação vs Payoff médio por estratégia")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / "coop_vs_payoff.png", dpi=300, bbox_inches="tight")
plt.show()